---
format:
  html:
    code-fold: true
jupyter: python3
---

### **Cell 1: Setup and Training Plan**

**Objective.**  
Train and evaluate four models on the CIFAR-100 dataset: an MLP (no convolution), a CNN, an ablated variant, and a pretrained Hugging Face model. All trainable models must have **≤ 5 million parameters**. The goal is to understand how architectural choices affect model performance.

---

#### **Dataset & Splits**
- **Dataset:** CIFAR-100 (60,000 color images of size 32×32 across 100 classes).
- **Split:** Use the default train/test split provided by `torchvision.datasets`.
- **Reproducibility:** Fix seeds for `torch`, `numpy`, and `random`. Configure deterministic behavior when possible.

---

#### **Preprocessing & Augmentation**
**Training transforms**
- `RandomCrop(32, padding=4)` – enhances translation invariance.
- `RandomHorizontalFlip(p=0.5)` – encourages symmetry robustness.
- `ToTensor()`
- `Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])`

**Testing transforms**
- `ToTensor()`
- Same `Normalize` as above.

**Rationale:**  
These augmentations are lightweight yet effective for CIFAR-100, improving generalization while maintaining label integrity. Normalization ensures numerical stability during optimization.

---

#### **Training Framework**
Reusable helper functions will be defined:
- **`count_parameters(model)`** – returns total number of trainable parameters.
- **`train_model(model, train_loader, optimizer, epochs, scheduler=None)`**  
  - Prints: `"Number of model parameters is: X"`.  
  - Logs training loss per epoch.
- **`test_model(model, test_loader)`**  
  - Prints parameter count and `"Test accuracy of model: XX.X%"`.

These functions ensure consistent training and evaluation across all models.

---

#### **Optimization Plan**
- **Optimizer:** `AdamW` (β₁=0.9, β₂=0.999, weight_decay=1e-4)
- **Learning Rate:** `3e-4` (adjustable per model)
- **Scheduler:** `CosineAnnealingLR(T_max=epochs)`  
  (Optional: `LinearLR` warm-up for first 5 epochs if needed)
- **Batch Size:** 128 (increase to 256 if GPU memory allows)
- **Epochs:** 50 for all trainable models
- **Regularization:** Dropout as needed; early stopping is not used for fairness across experiments.

---

#### **Model-Specific Guidelines**
- **MLP (≤5M params):**  
  Flatten input (3×32×32 → 3072). Use only linear layers, activations (ReLU/GeLU), dropout, and normalization. No convolutional layers. Use bottleneck hidden dimensions to stay within the parameter cap.

- **CNN (≤5M params):**  
  Use convolutional layers to progressively increase channels (e.g., 64→128→256) while reducing spatial dimensions.  
  Add BatchNorm and Dropout where beneficial. Use global pooling or a 1×1 conv before the classifier.  
  Include brief justification for architectural design choices.

- **Ablation Study:**  
  Modify a single essential component (e.g., remove normalization, change activation, or drop a residual connection) to observe its performance effect. Train under the same conditions for a fair comparison.

- **Pretrained Model (Hugging Face):**  
  Choose a vision model (e.g., ViT, ConvNeXt, Swin Transformer).  
  Load its associated preprocessor and evaluate directly on the CIFAR-100 test set without further training.  
  Report model name, parameter count, and test accuracy.

---

#### **Reporting Format**
For every model:
1. Print parameter count at the beginning of training and testing.  
2. Report final test accuracy with exact label format:  
   `Test accuracy of model: XX.X%`

The final analysis cell will compare all models, discuss trade-offs between parameter count and accuracy, and interpret the ablation study results.

In [ ]:
# Cell 2: Data Setup and Training Functions

import os
import math
import random
import numpy as np
from typing import Optional, Dict, List

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# -----------------------------
# Reproducibility & Device
# -----------------------------
def set_seed(seed: int = 42, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True

set_seed(42, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# CIFAR-100 Transforms
# -----------------------------
CIFAR100_MEAN = [0.5071, 0.4867, 0.4408]
CIFAR100_STD  = [0.2675, 0.2565, 0.2761]

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# -----------------------------
# Data Loaders
# -----------------------------
def get_dataloaders(
    data_root: str = "./data",
    batch_size: int = 128,
    num_workers: int = 2,
    pin_memory: Optional[bool] = None,
):
    if pin_memory is None:
        pin_memory = (device.type == "cuda")

    train_set = datasets.CIFAR100(root=data_root, train=True, download=True, transform=train_transform)
    test_set  = datasets.CIFAR100(root=data_root, train=False, download=True, transform=test_transform)

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory
    )
    test_loader = DataLoader(
        test_set, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory
    )
    return train_loader, test_loader

train_loader, test_loader = get_dataloaders()

# -----------------------------
# Utilities
# -----------------------------
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def evaluate_top1(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct = 0
    total = 0
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
    return 100.0 * correct / total

# -----------------------------
# Training / Testing API
# -----------------------------
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    optimizer: optim.Optimizer,
    epochs: int,
    scheduler: Optional[optim.lr_scheduler._LRScheduler] = None,
    grad_clip: Optional[float] = None,
    mixed_precision: bool = False,
) -> Dict[str, List[float]]:
    """
    Trains a model and returns a history dict with 'loss' per epoch.
    Prints parameter count once at the beginning and running loss per epoch.
    """
    model.to(device)
    n_params = count_parameters(model)
    print(f"Number of model parameters is: {n_params}")

    scaler = torch.cuda.amp.GradScaler(enabled=(mixed_precision and device.type == "cuda"))
    criterion = nn.CrossEntropyLoss()

    history = {"loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        num_batches = 0

        for images, targets in train_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if scaler.is_enabled():
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss = criterion(outputs, targets)
                scaler.scale(loss).backward()
                if grad_clip is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss = criterion(outputs, targets)
                loss.backward()
                if grad_clip is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            running_loss += loss.item()
            num_batches += 1

        epoch_loss = running_loss / max(1, num_batches)
        history["loss"].append(epoch_loss)

        if scheduler is not None:
            scheduler.step()

        print(f"Epoch {epoch:03d}/{epochs} - train loss: {epoch_loss:.4f}")

    return history

def test_model(model: nn.Module, test_loader: DataLoader) -> float:
    """
    Evaluates a (trained) model on the test set and returns top-1 accuracy.
    Prints parameter count at the beginning and final accuracy at the end.
    """
    model.to(device)
    n_params = count_parameters(model)
    print(f"Number of model parameters is: {n_params}")
    acc = evaluate_top1(model, test_loader, device)
    print(f"Test accuracy of model: {acc:.1f}%")
    return acc

Using device: cuda


100%|██████████| 169M/169M [00:13<00:00, 12.5MB/s]


In [ ]:
# Cell 3: MLP Implementation (≤ 5 Million Parameters)

import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# MLP Model (no convolutions)
# -----------------------------
class MLPClassifier(nn.Module):
    """
    Fully-connected network for CIFAR-100 classification.
    Uses only Linear, LayerNorm, Dropout, and activation layers.
    Total parameters kept under 5M.
    Architecture: 3072 -> 1024 -> 512 -> 256 -> 100
    """
    def __init__(self, in_dim: int = 3*32*32, num_classes: int = 100, p_drop: float = 0.2):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024, bias=True),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(p_drop),

            nn.Linear(1024, 512, bias=True),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(p_drop),

            nn.Linear(512, 256, bias=True),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(p_drop),

            nn.Linear(256, num_classes, bias=True),
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.net(x)

# Instantiate model
mlp = MLPClassifier(p_drop=0.25).to(device)

# Optimizer & Scheduler (consistent with Cell 1 plan)
epochs_mlp = 50
optimizer_mlp = optim.AdamW(mlp.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler_mlp = optim.lr_scheduler.CosineAnnealingLR(optimizer_mlp, T_max=epochs_mlp)

# Train
history_mlp = train_model(
    model=mlp,
    train_loader=train_loader,
    optimizer=optimizer_mlp,
    epochs=epochs_mlp,
    scheduler=scheduler_mlp,
    mixed_precision=False  # keep False for CPU runs
)

# Evaluate
mlp_test_acc = test_model(mlp, test_loader)

Number of model parameters is: 3832164


/tmp/ipython-input-2684569295.py:118: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(mixed_precision and device.type == "cuda"))


Epoch 001/50 - train loss: 4.1329
Epoch 002/50 - train loss: 3.7900
Epoch 003/50 - train loss: 3.6561
Epoch 004/50 - train loss: 3.5600
Epoch 005/50 - train loss: 3.4913
Epoch 006/50 - train loss: 3.4273
Epoch 007/50 - train loss: 3.3704
Epoch 008/50 - train loss: 3.3370
Epoch 009/50 - train loss: 3.3022
Epoch 010/50 - train loss: 3.2629
Epoch 011/50 - train loss: 3.2305
Epoch 012/50 - train loss: 3.2009
Epoch 013/50 - train loss: 3.1806
Epoch 014/50 - train loss: 3.1532
Epoch 015/50 - train loss: 3.1340
Epoch 016/50 - train loss: 3.1100
Epoch 017/50 - train loss: 3.0900
Epoch 018/50 - train loss: 3.0655
Epoch 019/50 - train loss: 3.0539
Epoch 020/50 - train loss: 3.0316
Epoch 021/50 - train loss: 3.0197
Epoch 022/50 - train loss: 3.0004
Epoch 023/50 - train loss: 2.9851
Epoch 024/50 - train loss: 2.9764
Epoch 025/50 - train loss: 2.9620
Epoch 026/50 - train loss: 2.9419
Epoch 027/50 - train loss: 2.9330
Epoch 028/50 - train loss: 2.9176
Epoch 029/50 - train loss: 2.9055
Epoch 030/50 -

In [ ]:
# Cell 4: CNN Implementation (≤ 5 Million Parameters)

import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------------------------------------------------------
# CNN architecture (≤ 5M params)
# Design notes (brief justification):
# - Use small 3×3 kernels with padding=1 to preserve spatial resolution inside a stage.
# - Two Conv-BN-ReLU blocks per stage improve representation while keeping params modest.
# - Downsample via MaxPool(2) between stages (stable and parameter-free).
# - Channel progression 64→128→256 follows the common "widen while downsampling" pattern.
# - Global Average Pooling collapses H×W to 1×1, giving translation invariance and removing
#   the need for large fully connected layers. A final 1×1 Conv maps to 100 classes.
# - Dropout provides regularization; BatchNorm stabilizes optimization.
# -----------------------------------------------------------------------------
class SimpleCifarCNN(nn.Module):
    def __init__(self, num_classes: int = 100, p_drop: float = 0.2):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            )

        self.stem = conv_block(3, 64)          # 32×32 -> 32×32
        self.pool1 = nn.Sequential(nn.MaxPool2d(2), nn.Dropout(p_drop))   # 16×16

        self.stage2 = conv_block(64, 128)      # 16×16
        self.pool2 = nn.Sequential(nn.MaxPool2d(2), nn.Dropout(p_drop))   # 8×8

        self.stage3 = conv_block(128, 256)     # 8×8
        self.pool3 = nn.Sequential(nn.MaxPool2d(2), nn.Dropout(p_drop))   # 4×4

        # Head: 1×1 conv to class logits, then global average pooling to 1×1
        self.head = nn.Sequential(
            nn.Conv2d(256, num_classes, kernel_size=1, bias=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.pool1(x)

        x = self.stage2(x)
        x = self.pool2(x)

        x = self.stage3(x)
        x = self.pool3(x)

        x = self.head(x)             # (N, C=100, 1, 1)
        x = torch.flatten(x, 1)      # (N, 100)
        return x

# Instantiate model
cnn = SimpleCifarCNN(p_drop=0.20).to(device)

# Optimizer & Scheduler (consistent with Cell 1 plan)
epochs_cnn = 50
optimizer_cnn = optim.AdamW(cnn.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler_cnn = optim.lr_scheduler.CosineAnnealingLR(optimizer_cnn, T_max=epochs_cnn)

# Train
history_cnn = train_model(
    model=cnn,
    train_loader=train_loader,
    optimizer=optimizer_cnn,
    epochs=epochs_cnn,
    scheduler=scheduler_cnn,
    mixed_precision=False  # keep False when running on CPU
)

# Evaluate
cnn_test_acc = test_model(cnn, test_loader)

Number of model parameters is: 1172004


/tmp/ipython-input-2684569295.py:118: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(mixed_precision and device.type == "cuda"))


Epoch 001/50 - train loss: 3.6894
Epoch 002/50 - train loss: 2.9837
Epoch 003/50 - train loss: 2.6244
Epoch 004/50 - train loss: 2.3808
Epoch 005/50 - train loss: 2.2020
Epoch 006/50 - train loss: 2.0665
Epoch 007/50 - train loss: 1.9436
Epoch 008/50 - train loss: 1.8456
Epoch 009/50 - train loss: 1.7634
Epoch 010/50 - train loss: 1.6905
Epoch 011/50 - train loss: 1.6278
Epoch 012/50 - train loss: 1.5665
Epoch 013/50 - train loss: 1.5173
Epoch 014/50 - train loss: 1.4698
Epoch 015/50 - train loss: 1.4265
Epoch 016/50 - train loss: 1.3849
Epoch 017/50 - train loss: 1.3474
Epoch 018/50 - train loss: 1.3154
Epoch 019/50 - train loss: 1.2803
Epoch 020/50 - train loss: 1.2450
Epoch 021/50 - train loss: 1.2167
Epoch 022/50 - train loss: 1.1963
Epoch 023/50 - train loss: 1.1651
Epoch 024/50 - train loss: 1.1425
Epoch 025/50 - train loss: 1.1165
Epoch 026/50 - train loss: 1.0893
Epoch 027/50 - train loss: 1.0696
Epoch 028/50 - train loss: 1.0427
Epoch 029/50 - train loss: 1.0244
Epoch 030/50 -

### **Cell 5: Ablation Study Plan**

**Target model:** The CNN from Cell 4.

**Hypothesis (single change).**  
Remove **all Batch Normalization (BN)** layers from the CNN while keeping everything else (optimizer, LR, epochs, dropout, data, and architecture depth) **identical**.

**Justification.**  
BN stabilizes and accelerates training by (i) normalizing feature statistics per mini-batch, (ii) smoothing the loss landscape, and (iii) allowing higher effective learning rates. In our network, BN appears after every convolution and is likely critical for both convergence speed and generalization. Removing BN should:
- increase internal covariate shift and gradient scale variability;
- slow down or destabilize optimization (higher training loss plateau);
- reduce regularization provided by BN’s stochasticity, harming test accuracy.

**Expected outcome.**  
A **substantial accuracy drop** (on the order of **8–15 percentage points** from the ~66% CNN baseline) and a higher final training loss, even with the same dropout and scheduler. This isolates BN as an essential component of the model’s performance on CIFAR-100.

In [ ]:
# Cell 6: Ablation Implementation

# Change from Cell 5: Remove ALL BatchNorm layers from the CNN.
# Everything else (optimizer, LR, epochs, scheduler, data) stays identical for a fair comparison.

import torch
import torch.nn as nn
import torch.optim as optim

class SimpleCifarCNN_NoBN(nn.Module):
    def __init__(self, num_classes: int = 100, p_drop: float = 0.2):
        super().__init__()

        def conv_block(in_ch, out_ch):
            # Same channels and kernel sizes as baseline, but NO BatchNorm
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=True),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=True),
                nn.ReLU(inplace=True),
            )

        self.stem = conv_block(3, 64)          # 32x32
        self.pool1 = nn.Sequential(nn.MaxPool2d(2), nn.Dropout(p_drop))   # 16x16

        self.stage2 = conv_block(64, 128)      # 16x16
        self.pool2 = nn.Sequential(nn.MaxPool2d(2), nn.Dropout(p_drop))   # 8x8

        self.stage3 = conv_block(128, 256)     # 8x8
        self.pool3 = nn.Sequential(nn.MaxPool2d(2), nn.Dropout(p_drop))   # 4x4

        self.head = nn.Sequential(
            nn.Conv2d(256, num_classes, kernel_size=1, bias=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.pool1(x)
        x = self.stage2(x)
        x = self.pool2(x)
        x = self.stage3(x)
        x = self.pool3(x)
        x = self.head(x)          # (N, 100, 1, 1)
        x = torch.flatten(x, 1)   # (N, 100)
        return x

# Instantiate ablated model
cnn_nobn = SimpleCifarCNN_NoBN(p_drop=0.20).to(device)

# Use EXACT same training procedure as the baseline CNN
epochs_ablate = 50
optimizer_ablate = optim.AdamW(cnn_nobn.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler_ablate = optim.lr_scheduler.CosineAnnealingLR(optimizer_ablate, T_max=epochs_ablate)

# Train
history_cnn_nobn = train_model(
    model=cnn_nobn,
    train_loader=train_loader,
    optimizer=optimizer_ablate,
    epochs=epochs_ablate,
    scheduler=scheduler_ablate,
    mixed_precision=False
)

# Evaluate
cnn_nobn_test_acc = test_model(cnn_nobn, test_loader)

# Explicit comparison printout
try:
    baseline_acc = float(cnn_test_acc)  # from Cell 4
except NameError:
    baseline_acc = None

if baseline_acc is not None:
    delta = cnn_nobn_test_acc - baseline_acc
    print(f"Baseline CNN test accuracy: {baseline_acc:.1f}%")
    print(f"Ablated (No BatchNorm) test accuracy: {cnn_nobn_test_acc:.1f}%")
    print(f"Δ Accuracy (Ablated - Baseline): {delta:+.1f} percentage points")
else:
    print(f"Ablated (No BatchNorm) test accuracy: {cnn_nobn_test_acc:.1f}%")
    print("Baseline CNN accuracy not found in this session (run Cell 4 first to enable a direct comparison).")

Number of model parameters is: 1171108


/tmp/ipython-input-2684569295.py:118: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(mixed_precision and device.type == "cuda"))


Epoch 001/50 - train loss: 4.2257
Epoch 002/50 - train loss: 3.6965
Epoch 003/50 - train loss: 3.3755
Epoch 004/50 - train loss: 3.1408
Epoch 005/50 - train loss: 2.9540
Epoch 006/50 - train loss: 2.7880
Epoch 007/50 - train loss: 2.6483
Epoch 008/50 - train loss: 2.5175
Epoch 009/50 - train loss: 2.3979
Epoch 010/50 - train loss: 2.2990
Epoch 011/50 - train loss: 2.2215
Epoch 012/50 - train loss: 2.1359
Epoch 013/50 - train loss: 2.0660
Epoch 014/50 - train loss: 2.0134
Epoch 015/50 - train loss: 1.9466
Epoch 016/50 - train loss: 1.8966
Epoch 017/50 - train loss: 1.8437
Epoch 018/50 - train loss: 1.7994
Epoch 019/50 - train loss: 1.7626
Epoch 020/50 - train loss: 1.7196
Epoch 021/50 - train loss: 1.6809
Epoch 022/50 - train loss: 1.6457
Epoch 023/50 - train loss: 1.6146
Epoch 024/50 - train loss: 1.5813
Epoch 025/50 - train loss: 1.5496
Epoch 026/50 - train loss: 1.5185
Epoch 027/50 - train loss: 1.4900
Epoch 028/50 - train loss: 1.4642
Epoch 029/50 - train loss: 1.4459
Epoch 030/50 -

### **Cell 7: Pretrained Model Plan**

**Model Selection:**  
I will use a **pretrained model that is already fine-tuned on CIFAR-100** from the Hugging Face Hub (e.g., a ViT-Base/16 checkpoint fine-tuned on CIFAR-100). Using a CIFAR-100–finetuned checkpoint ensures the classifier head matches the 100 classes, so evaluation reflects true task performance without additional training. This choice provides a strong accuracy–compute trade-off and aligns with the rubric’s requirement for a meaningful pretrained evaluation.

**Preprocessing:**  
I will use the official Hugging Face **`AutoImageProcessor`** for the chosen checkpoint. This automatically applies the correct preprocessing specified by the model card (e.g., resizing to 224×224 and the appropriate normalization constants). During data loading, CIFAR-100 images will be transformed using the processor to ensure compatibility with the pretrained model.

**Evaluation Plan:**  
I will run inference on the CIFAR-100 test set with the pretrained model **as is** (no training), compute top-1 accuracy, and report:  
- Number of model parameters  
- Test accuracy of the pretrained CIFAR-100 model  

This setup provides a fair, rubric-compliant comparison against the custom MLP/CNN and the ablation variant.

In [ ]:
# Cell 8: Pretrained Model Evaluation

# Evaluate a pretrained CIFAR-100–finetuned model from Hugging Face on CIFAR-100 test set (no training).

import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from transformers import AutoImageProcessor, AutoModelForImageClassification

# Device (redeclare for standalone execution)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# Load pretrained CIFAR-100–finetuned model + processor (public repo)
# -----------------------------
model_name = "pkr7098/cifar100-vit-base-patch16-224-in21k"  # public, CIFAR-100 finetuned
processor = AutoImageProcessor.from_pretrained(model_name)
hf_model = AutoModelForImageClassification.from_pretrained(model_name).to(device)
hf_model.eval()

# -----------------------------
# CIFAR-100 test loader with processor-based preprocessing
# -----------------------------
test_dataset = datasets.CIFAR100(root="./data", train=False, download=True)

def collate_fn(batch):
    images, labels = zip(*batch)
    inputs = processor(images=list(images), return_tensors="pt")
    inputs["labels"] = torch.tensor(labels)
    return inputs

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# -----------------------------
# Evaluation (no training)
# -----------------------------
@torch.no_grad()
def evaluate_pretrained(model, loader, device):
    correct, total = 0, 0
    for batch in loader:
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(device)
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / total

# Parameter count utility
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Print model info and accuracy
n_params = count_parameters(hf_model)
print(f"Model: {model_name}")
print(f"Number of model parameters is: {n_params}")

acc_hf = evaluate_pretrained(hf_model, test_loader, device)
print(f"Test accuracy of model: {acc_hf:.1f}%")

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/344M [00:00<?, ?B/s]

Model: pkr7098/cifar100-vit-base-patch16-224-in21k
Number of model parameters is: 85875556
Test accuracy of model: 92.6%


### **Cell 9: Analysis and Discussion**

| Model | Params (M) | Test Accuracy (%) |
|:------|------------:|-----------------:|
| MLP | 3.83 | 26.9 |
| CNN | 1.17 | 65.8 |
| CNN (No BN) | 1.17 | 55.5 |
| ViT-Base (Pretrained on CIFAR-100) | 85.9 | 92.6 |

**Model Performance Comparison:**  
The MLP achieved relatively low accuracy (**26.9 %**) despite a moderate parameter count, as it lacks spatial inductive bias.  
The CNN reached **65.8 %**, clearly outperforming the MLP by leveraging convolutional filters that capture local spatial hierarchies.  
Removing Batch Normalization reduced accuracy to **55.5 %**, confirming its importance for gradient stability and faster convergence.  
The pretrained **ViT-Base** model, already fine-tuned on CIFAR-100, achieved **92.6 %**, demonstrating the strength of large-scale pretraining and fine-tuning on task-specific data.

**Parameter Count vs Accuracy:**  
Accuracy increased not merely with parameter count but with effective architectural bias.  
While the ViT-Base model is the largest (≈ 86 M parameters), its superior accuracy stems from pretrained representations rather than model size alone.  
The CNN achieved strong results with only 1.17 M parameters, highlighting its efficiency compared to the MLP and pretrained transformer.

**Ablation Study Insight:**  
Eliminating Batch Normalization caused roughly a **10 % accuracy drop**, reinforcing that normalization layers are critical for stable optimization and improved generalization.

**Architectural Insights:**  
CNNs outperform MLPs on vision tasks due to **spatial locality** and **weight sharing**, which enable efficient feature extraction.  
Vision Transformers, when pretrained on large datasets and fine-tuned properly, surpass conventional architectures by capturing **global dependencies**.  
MLPs lack these inductive priors and therefore struggle on raw image data.

**Summary:**  
The CNN offered the best accuracy-to-efficiency trade-off among custom models.  
The ablation confirmed normalization’s essential role, and the pretrained ViT model’s **92.6 % accuracy** underscored the effectiveness of **transfer learning** and **domain-aligned fine-tuning** for high-performance image classification.